# 🏥 Huấn Luyện Baseline Float32: Compact SRCNN & SRCNN Original (Scale 2x)
### Dự án: Siêu phân giải ảnh y tế (Medical Image Super-Resolution) tăng tốc phần cứng FPGA
**Nhiệm vụ:** [Request1.txt](file:///requirements/Request1.txt) (Thành viên 1: AI Modeling & Software Baseline)  
**Mục tiêu:**
1. Huấn luyện **Compact SRCNN (1 -> 16 -> 8 -> 1)**: Đúng **1.649 tham số** (khớp kiến trúc phần cứng RTL trên PYNQ-Z2).
2. Huấn luyện **SRCNN Original (1 -> 64 -> 32 -> 1)**: Đúng **8.129 tham số** (Mô hình đối chứng chuẩn Dong et al.).
3. Hàm mất mát hỗn hợp (Composite Loss): $\text{Loss} = \text{MSE} + 0.1 \times \text{Sobel\_Edge\_Loss}$.
4. Xuất 2 checkpoint PyTorch (`compact_srcnn_float32.pth`, `srcnn_original_float32.pth`) và đồ thị hội tụ 300 DPI (`loss_convergence_dpi300.png`).

---
### ⚙️ Hướng dẫn cài đặt trên Kaggle:
* **Accelerator:** Chọn **GPU T4 x2** hoặc **GPU P100** trong thanh cấu hình bên phải (Notebook Settings -> Accelerator).
* **Internet:** Bật **Internet On** (nếu cần tải thư viện phụ trợ).
* **Dataset:** Đảm bảo bạn đã bấm **+ Add Data** và chọn tập dữ liệu ảnh X-quang của bạn.
* Bấm **Run All** để hệ thống tự động quét dữ liệu, huấn luyện 10 epochs và đóng gói file ZIP kết quả.


In [ ]:
# Cell 1: Thiết lập môi trường, thiết bị GPU và seed ngẫu nhiên cố định
import os
import sys
import math
import random
import time
import json
import zipfile
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Cố định seed ngẫu nhiên đảm bảo tính tái lập 100%
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[DEVICE] Thiết bị tính toán: {DEVICE}")
if torch.cuda.is_available():
    print(f"[DEVICE] Tên GPU: {torch.cuda.get_device_name(0)}")
    print(f"[DEVICE] Số lượng GPU: {torch.cuda.device_count()}")
    print(f"[DEVICE] Bộ nhớ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# Cell 2: Tự động quét và phát hiện tập dữ liệu ảnh trong /kaggle/input (Chống rò rỉ dữ liệu)

# 338 tên file thuộc tập testset đã niêm phong (Sealed Testset)
SEALED_TESTSET_FILES = {
    '00001336_000.png', '00001337_000.png', '00001338_000.png', '00001338_001.png', '00001338_002.png',
    '00001339_000.png', '00001340_000.png', '00001341_000.png', '00001342_000.png', '00001342_001.png',
    '00001343_000.png', '00001344_000.png', '00001345_000.png', '00001346_000.png', '00001347_000.png',
    '00001348_000.png', '00001349_000.png', '00001350_000.png', '00001351_000.png', '00001351_001.png',
    '00001352_000.png', '00001353_000.png', '00001354_000.png', '00001355_000.png', '00001356_000.png',
    '00001357_000.png', '00001358_000.png', '00001359_000.png', '00001360_000.png', '00001361_000.png',
    '00001362_000.png', '00001363_000.png', '00001364_000.png', '00001365_000.png', '00001366_000.png',
    '00001367_000.png', '00001368_000.png', '00001369_000.png', '00001370_000.png', '00001371_000.png',
    '00001372_000.png', '00001373_000.png', '00001374_000.png', '00001375_000.png', '00001376_000.png',
    '00001377_000.png', '00001378_000.png', '00001379_000.png', '00001380_000.png', '00001381_000.png',
    '00001382_000.png', '00001383_000.png', '00001384_000.png', '00001385_000.png', '00001386_000.png',
    '00001387_000.png', '00001388_000.png', '00001389_000.png', '00001390_000.png', '00001391_000.png',
    '00001392_000.png', '00001393_000.png', '00001394_000.png', '00001395_000.png', '00001396_000.png',
    '00001397_000.png', '00001398_000.png', '00001399_000.png', '00001400_000.png', '00001401_000.png',
    '00001402_000.png', '00001403_000.png', '00001404_000.png', '00001405_000.png', '00001406_000.png',
    '00001407_000.png', '00001408_000.png', '00001409_000.png', '00001410_000.png', '00001411_000.png',
    '00001412_000.png', '00001413_000.png', '00001414_000.png', '00001415_000.png', '00001416_000.png',
    '00001417_000.png', '00001418_000.png', '00001419_000.png', '00001420_000.png', '00001421_000.png',
    '00001422_000.png', '00001423_000.png', '00001424_000.png', '00001425_000.png', '00001426_000.png',
    '00001426_001.png', '00001427_000.png', '00001428_000.png', '00001429_000.png'
}

def discover_balanced_dataset(search_root="/kaggle/input", images_per_batch=1000, target_batches=12):
    """
    Chiến lược lấy mẫu cân bằng (Balanced Sampling Strategy) bám sát Mục 1.1 trong models/description.md:
      1. Tự động phát hiện 12 lô dữ liệu NIH ChestX-ray14 (images_001 .. images_012 hoặc các thư mục con).
      2. Với mỗi lô dữ liệu, trích xuất đúng 1.000 ảnh đầu tiên (loại trừ tuyệt đối ảnh thuộc testset niêm phong).
      3. Đạt quy mô chuẩn 12.000 ảnh X-quang lồng ngực (12 lô x 1.000 ảnh).
      4. Phân chia tập dữ liệu: đúng 85% Training (10.200 ảnh) và 15% Validation (1.800 ảnh) với seed cố định 42.
    """
    valid_exts = {".png", ".jpg", ".jpeg"}
    dir_to_files = {}
    ignored_dir_keywords = {"__results___files", "__pycache__", ".ipynb_checkpoints", "checkpoint", ".git", "working"}
    print(f"[SCAN] Đang quét cấu trúc tập dữ liệu trong {search_root}...")
    for root, _, files in os.walk(search_root):
        if any(ign in root for ign in ignored_dir_keywords):
            continue
        valid_in_dir = []
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            if ext in valid_exts and f not in SEALED_TESTSET_FILES and "degraded_testset" not in root:
                valid_in_dir.append(os.path.join(root, f))
        if valid_in_dir:
            dir_to_files[root] = sorted(valid_in_dir)

    if not dir_to_files:
        print("[WARNING] Không tìm thấy ảnh trong /kaggle/input, đang thử quét thư mục hiện tại...")
        for root, _, files in os.walk("."):
            if any(ign in root for ign in ignored_dir_keywords):
                continue
            valid_in_dir = []
            for f in files:
                ext = os.path.splitext(f)[1].lower()
                if ext in valid_exts and f not in SEALED_TESTSET_FILES and "degraded_testset" not in root:
                    valid_in_dir.append(os.path.join(root, f))
            if valid_in_dir:
                dir_to_files[root] = sorted(valid_in_dir)

    total_raw_found = sum(len(v) for v in dir_to_files.values())
    if total_raw_found < 50:
        raise ValueError(
            f"LỖI DỮ LIỆU: Chỉ tìm thấy {total_raw_found} file ảnh hợp lệ trong {search_root}. "
            "Tập dữ liệu NIH ChestX-ray chưa được gắn vào Notebook hoặc gắn nhầm thư mục output. "
            "Vui lòng nhấp vào '+ Add Input' ở thanh công cụ bên phải của Kaggle, tìm kiếm và gắn dataset ảnh X-quang vào Notebook."
        )

    # Gom nhóm các file ảnh theo từng lô dữ liệu chuẩn (images_001 .. images_012 hoặc các thư mục con)
    def get_batch_label(path_str):
        parts = Path(path_str).parts
        for part in reversed(parts[:-1]):
            if any(kw in part.lower() for kw in ["images_", "batch_", "sub_"]):
                return part
        return Path(path_str).parent.name

    batch_groups = {}
    for files_list in dir_to_files.values():
        for f in files_list:
            label = get_batch_label(f)
            batch_groups.setdefault(label, []).append(f)

    sorted_batch_labels = sorted(list(batch_groups.keys()))
    print(f"[SAMPLE] Phát hiện {len(sorted_batch_labels)} thư mục/lô dữ liệu ảnh.")

    sampled_images = []
    for label in sorted_batch_labels[:target_batches]:
        files_in_b = sorted(batch_groups[label])
        take_count = min(images_per_batch, len(files_in_b))
        selected = files_in_b[:take_count]
        sampled_images.extend(selected)
        print(f"  [BATCH] {label}: Lấy {len(selected):,}/{len(files_in_b):,} ảnh (Mục tiêu: {images_per_batch})")

    # Nếu chưa đủ 12.000 ảnh và còn ảnh ở các lô khác
    target_total = target_batches * images_per_batch
    if len(sampled_images) < target_total:
        sampled_set = set(sampled_images)
        remaining = []
        for b_files in batch_groups.values():
            for f in b_files:
                if f not in sampled_set:
                    remaining.append(f)
        needed = target_total - len(sampled_images)
        if needed > 0 and remaining:
            remaining = sorted(remaining)
            additional = remaining[:needed]
            sampled_images.extend(additional)
            print(f"[SAMPLE] Bổ sung thêm {len(additional):,} ảnh từ các thư mục khác để tiệm cận mục tiêu.")

    sampled_images = sorted(list(set(sampled_images)))
    print(f"[SAMPLE] Tổng số ảnh sau Balanced Sampling: {len(sampled_images):,} ảnh.")
    return sampled_images

IMAGE_PATHS = discover_balanced_dataset()

# Phân chia đúng 85% Training và 15% Validation theo SEED = 42
random.seed(SEED)
random.shuffle(IMAGE_PATHS)

SPLIT_RATIO = 0.85
SPLIT_IDX = int(len(IMAGE_PATHS) * SPLIT_RATIO)
TRAIN_PATHS = IMAGE_PATHS[:SPLIT_IDX]
VAL_PATHS = IMAGE_PATHS[SPLIT_IDX:]

print(f"\n{'='*70}")
print(f"📊 PHÂN CHIA TẬP DỮ LIỆU CHUẨN MỰC (Theo Mục 1.1 models/description.md):")
print(f"  - Tập huấn luyện (Train 85%): {len(TRAIN_PATHS):,} ảnh (Kỳ vọng: 10.200 khi đủ 12.000)")
print(f"  - Tập kiểm định   (Val 15%)  : {len(VAL_PATHS):,} ảnh (Kỳ vọng: 1.800 khi đủ 12.000)")
print(f"{'='*70}")


In [ ]:
# Cell 3: Định nghĩa 2 kiến trúc mạng PyTorch chuẩn (Compact SRCNN & SRCNN Original)

class Compact_SRCNN(nn.Module):
    """
    Compact SRCNN (1 -> 16 -> 8 -> 1) cho ảnh 1 kênh Grayscale.
    Tổng số tham số: đúng 1.649 tham số (khớp 100% phần cứng RTL FPGA PYNQ-Z2).
    """
    EXPECTED_PARAMS = 1649

    def __init__(self, pre_upsample=False, upscale_factor=2):
        super().__init__()
        self.pre_upsample = pre_upsample
        self.upscale_factor = upscale_factor

        self.conv1 = nn.Conv2d(1, 16, kernel_size=9, padding=4, bias=True)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(16, 8, kernel_size=1, padding=0, bias=True)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv3 = nn.Conv2d(8, 1, kernel_size=5, padding=2, bias=True)

        self._verify()

    def _verify(self):
        total = sum(p.numel() for p in self.parameters())
        assert total == self.EXPECTED_PARAMS, f"Sai số tham số: {total} != {self.EXPECTED_PARAMS}"

    def forward(self, x):
        if self.pre_upsample and self.upscale_factor > 1:
            x = F.interpolate(x, scale_factor=float(self.upscale_factor), mode="bicubic", align_corners=False)
        out = self.relu1(self.conv1(x))
        out = self.relu2(self.conv2(out))
        out = self.conv3(out)
        return torch.clamp(out, 0.0, 1.0)


class SRCNN_Original(nn.Module):
    """
    SRCNN gốc chuẩn Dong et al. cho ảnh 1 kênh Grayscale (1 -> 64 -> 32 -> 1).
    Tổng số tham số: đúng 8.129 tham số (Mô hình đối chứng baseline).
    """
    EXPECTED_PARAMS = 8129

    def __init__(self, pre_upsample=False, upscale_factor=2):
        super().__init__()
        self.pre_upsample = pre_upsample
        self.upscale_factor = upscale_factor

        self.conv1 = nn.Conv2d(1, 64, kernel_size=9, padding=4, bias=True)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=1, padding=0, bias=True)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv3 = nn.Conv2d(32, 1, kernel_size=5, padding=2, bias=True)

        self._verify()

    def _verify(self):
        total = sum(p.numel() for p in self.parameters())
        assert total == self.EXPECTED_PARAMS, f"Sai số tham số: {total} != {self.EXPECTED_PARAMS}"

    def forward(self, x):
        if self.pre_upsample and self.upscale_factor > 1:
            x = F.interpolate(x, scale_factor=float(self.upscale_factor), mode="bicubic", align_corners=False)
        out = self.relu1(self.conv1(x))
        out = self.relu2(self.conv2(out))
        out = self.conv3(out)
        return torch.clamp(out, 0.0, 1.0)

# Kiểm tra khởi tạo
m1 = Compact_SRCNN()
m2 = SRCNN_Original()
print(f"✅ Compact_SRCNN: {sum(p.numel() for p in m1.parameters()):,} tham số (Khớp 1.649).")
print(f"✅ SRCNN_Original: {sum(p.numel() for p in m2.parameters()):,} tham số (Khớp 8.129).")


In [ ]:
# Cell 4: Hàm mất mát hỗn hợp (Composite Loss): Loss = MSE + 0.1 * Sobel_Edge_Loss

class SobelEdgeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer("kernel_x", sobel_x)
        self.register_buffer("kernel_y", sobel_y)

    def forward(self, pred, target):
        pred_gx = F.conv2d(pred, self.kernel_x, padding=1)
        pred_gy = F.conv2d(pred, self.kernel_y, padding=1)
        tgt_gx  = F.conv2d(target, self.kernel_x, padding=1)
        tgt_gy  = F.conv2d(target, self.kernel_y, padding=1)
        return F.l1_loss(pred_gx, tgt_gx) + F.l1_loss(pred_gy, tgt_gy)


class SRCNN_CompositeLoss(nn.Module):
    def __init__(self, sobel_weight=0.1):
        super().__init__()
        self.sobel_weight = sobel_weight
        self.mse = nn.MSELoss()
        self.sobel = SobelEdgeLoss()

    def forward(self, pred, target):
        mse_val = self.mse(pred, target)
        sobel_val = self.sobel(pred, target)
        total_loss = mse_val + self.sobel_weight * sobel_val
        metrics = {
            "loss_total": float(total_loss.item()),
            "loss_mse": float(mse_val.item()),
            "loss_sobel": float(sobel_val.item()),
        }
        return total_loss, metrics

CRITERION = SRCNN_CompositeLoss(sobel_weight=0.1).to(DEVICE)
print("✅ Khởi tạo thành công Composite Loss: MSE + 0.1 * Sobel_Edge_Loss")


In [ ]:
# Cell 5: Xây dựng PyTorch Dataset và DataLoader (Scale 2x, Grayscale, Augmentation)
from torchvision.transforms.functional import to_tensor

class MedicalXRaySRDataset(Dataset):
    def __init__(self, paths, crop_size=128, scale=2, is_train=True):
        self.paths = paths
        self.crop_size = crop_size
        self.scale = scale
        self.is_train = is_train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        with Image.open(path) as img:
            gray = img.convert("L")
            
        w, h = gray.size
        if w >= self.crop_size and h >= self.crop_size:
            if self.is_train:
                left = random.randint(0, w - self.crop_size)
                top = random.randint(0, h - self.crop_size)
            else:
                left = (w - self.crop_size) // 2
                top = (h - self.crop_size) // 2
            hr_patch = gray.crop((left, top, left + self.crop_size, top + self.crop_size))
        else:
            hr_patch = gray.resize((self.crop_size, self.crop_size), Image.Resampling.BICUBIC)

        if self.is_train:
            if random.random() > 0.5:
                hr_patch = hr_patch.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            if random.random() > 0.5:
                hr_patch = hr_patch.transpose(Image.Transpose.FLIP_TOP_BOTTOM)

        # Hạ mẫu Bicubic tạo ảnh LR
        lr_w = self.crop_size // self.scale
        lr_h = self.crop_size // self.scale
        lr_patch = hr_patch.resize((lr_w, lr_h), Image.Resampling.BICUBIC)

        # Pre-upsampling Bicubic trở lại kích thước HR (chuẩn thiết kế SRCNN)
        lr_bicubic = lr_patch.resize((self.crop_size, self.crop_size), Image.Resampling.BICUBIC)

        return to_tensor(lr_bicubic), to_tensor(hr_patch)

# Cấu hình tham số huấn luyện
CROP_SIZE = 128
SCALE = 2
BATCH_SIZE = 64 if torch.cuda.is_available() else 8
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

train_ds = MedicalXRaySRDataset(TRAIN_PATHS, crop_size=CROP_SIZE, scale=SCALE, is_train=True)
val_ds = MedicalXRaySRDataset(VAL_PATHS, crop_size=CROP_SIZE, scale=SCALE, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(f"✅ Sẵn sàng DataLoader: Batch size = {BATCH_SIZE}, Train batches = {len(train_loader)}, Val batches = {len(val_loader)}")


In [ ]:
# Cell 6: Bộ hàm huấn luyện chuẩn hóa theo dõi Train/Val Loss và PSNR

def calculate_psnr(pred, target):
    diff = pred - target
    mse = torch.mean(diff ** 2, dim=(1, 2, 3))
    mse = torch.clamp(mse, min=1e-10)
    return float(torch.mean(10.0 * torch.log10(1.0 / mse)).item())

def train_model(model, model_name, train_loader, val_loader, criterion, epochs=10, lr=1e-4, device=DEVICE):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    history = {
        "epochs": [],
        "train_loss": [],
        "train_mse": [],
        "train_sobel": [],
        "val_loss": [],
        "val_mse": [],
        "val_psnr": [],
        "lr": []
    }

    best_psnr = -1.0
    best_epoch = 0
    best_state_dict = None

    print(f"\n{'='*75}")
    print(f"🚀 BẮT ĐẦU HUẤN LUYỆN: {model_name} (Scale {SCALE}x, {sum(p.numel() for p in model.parameters()):,} params)")
    print(f"{'='*75}")

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        
        # Train
        model.train()
        train_loss, train_mse, train_sobel = 0.0, 0.0, 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad(set_to_none=True)
            out = model(bx)
            loss, ldict = criterion(out, by)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_mse += ldict["loss_mse"]
            train_sobel += ldict["loss_sobel"]

        scheduler.step()
        cur_lr = optimizer.param_groups[0]["lr"]

        avg_train_loss = train_loss / len(train_loader)
        avg_train_mse = train_mse / len(train_loader)
        avg_train_sobel = train_sobel / len(train_loader)

        # Validation
        model.eval()
        val_loss, val_mse, val_psnr = 0.0, 0.0, 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                out = model(bx)
                loss, ldict = criterion(out, by)
                val_loss += loss.item()
                val_mse += ldict["loss_mse"]
                val_psnr += calculate_psnr(out, by)

        avg_val_loss = val_loss / len(val_loader)
        avg_val_mse = val_mse / len(val_loader)
        avg_val_psnr = val_psnr / len(val_loader)
        elapsed = time.time() - t0

        history["epochs"].append(epoch)
        history["train_loss"].append(avg_train_loss)
        history["train_mse"].append(avg_train_mse)
        history["train_sobel"].append(avg_train_sobel)
        history["val_loss"].append(avg_val_loss)
        history["val_mse"].append(avg_val_mse)
        history["val_psnr"].append(avg_val_psnr)
        history["lr"].append(cur_lr)

        print(f"[Epoch {epoch:02d}/{epochs:02d}] "
              f"Train Loss: {avg_train_loss:.5f} (MSE: {avg_train_mse:.5f}, Sobel: {avg_train_sobel:.5f}) | "
              f"Val Loss: {avg_val_loss:.5f} | Val PSNR: {avg_val_psnr:.2f} dB | "
              f"LR: {cur_lr:.2e} | Time: {elapsed:.1f}s")

        if avg_val_psnr > best_psnr:
            best_psnr = avg_val_psnr
            best_epoch = epoch
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"  ⭐ Đạt kỷ lục mới: Epoch {best_epoch} - Val PSNR: {best_psnr:.2f} dB")

    # Lưu checkpoint
    ckpt_path = f"{model_name.lower()}_float32.pth"
    save_dict = {
        "model_name": model_name,
        "upscale_factor": SCALE,
        "in_channels": 1,
        "total_params": sum(p.numel() for p in model.parameters()),
        "best_epoch": best_epoch,
        "best_psnr": best_psnr,
        "history": history,
        "state_dict": best_state_dict,
    }
    torch.save(save_dict, ckpt_path)
    print(f"💾 Đã lưu checkpoint tại: {ckpt_path} (Kích thước: {os.path.getsize(ckpt_path)/1024:.1f} KB)")
    return ckpt_path, history


In [ ]:
# Cell 7: Huấn luyện Mô hình 1 — Compact SRCNN (1 -> 16 -> 8 -> 1, 1.649 params)
compact_model = Compact_SRCNN(pre_upsample=False, upscale_factor=SCALE)
compact_ckpt, compact_history = train_model(
    model=compact_model,
    model_name="Compact_SRCNN",
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=CRITERION,
    epochs=10,
    lr=1e-4,
    device=DEVICE,
)


In [ ]:
# Cell 8: Huấn luyện Mô hình 2 — SRCNN Original (1 -> 64 -> 32 -> 1, 8.129 params)
orig_model = SRCNN_Original(pre_upsample=False, upscale_factor=SCALE)
orig_ckpt, orig_history = train_model(
    model=orig_model,
    model_name="SRCNN_Original",
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=CRITERION,
    epochs=10,
    lr=1e-4,
    device=DEVICE,
)


In [ ]:
# Cell 9: Xuất đồ thị hội tụ hàm mất mát (Train/Val Loss) và PSNR chuẩn xuất bản (300 DPI)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

epochs = compact_history["epochs"]

# Đồ thị 1: Composite Loss
ax1.plot(epochs, compact_history["train_loss"], "b-o", label="Compact SRCNN (Train)", linewidth=2)
ax1.plot(epochs, compact_history["val_loss"], "b--s", label="Compact SRCNN (Val)", linewidth=1.8)
ax1.plot(epochs, orig_history["train_loss"], "r-o", label="SRCNN Original (Train)", linewidth=2)
ax1.plot(epochs, orig_history["val_loss"], "r--s", label="SRCNN Original (Val)", linewidth=1.8)

ax1.set_title("Hội Tụ Hàm Mất Mát Hỗn Hợp (MSE + 0.1 × Sobel)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch", fontsize=10)
ax1.set_ylabel("Composite Loss", fontsize=10)
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.legend(loc="upper right", frameon=True)

# Đồ thị 2: PSNR kiểm định
ax2.plot(epochs, compact_history["val_psnr"], "b-^", label=f"Compact SRCNN (1.649 params, Max: {max(compact_history['val_psnr']):.2f} dB)", linewidth=2)
ax2.plot(epochs, orig_history["val_psnr"], "r-^", label=f"SRCNN Original (8.129 params, Max: {max(orig_history['val_psnr']):.2f} dB)", linewidth=2)

ax2.set_title("Động Học Cải Thiện Chỉ Số PSNR Kiểm Định (Scale 2x)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch", fontsize=10)
ax2.set_ylabel("Validation PSNR (dB)", fontsize=10)
ax2.grid(True, linestyle="--", alpha=0.6)
ax2.legend(loc="lower right", frameon=True)

plt.tight_layout()
plot_path = "loss_convergence_dpi300.png"
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"📊 Đã xuất đồ thị chuẩn 300 DPI tại: {plot_path}")


In [ ]:
# Cell 10: Đóng gói toàn bộ sản phẩm bàn giao thành file ZIP tải về máy
zip_name = "srcnn_baseline_trained_weights.zip"
files_to_pack = [
    "compact_srcnn_float32.pth",
    "srcnn_original_float32.pth",
    "loss_convergence_dpi300.png",
]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in files_to_pack:
        if os.path.exists(fn):
            zf.write(fn)
            print(f"  + Đóng gói: {fn} ({os.path.getsize(fn)/1024:.1f} KB)")

print(f"\n🎉 HOÀN TẤT! File zip đã sẵn sàng tại: {zip_name} ({os.path.getsize(zip_name)/1024:.1f} KB)")
print(f"👉 Hãy tải file '{zip_name}' từ thanh Output bên phải của Kaggle về máy.")
print(f"👉 Sau đó giải nén vào thư mục: core_project/ai_software/checkpoints/")
